# Dataset EDA — Amazon Beauty, Amazon Video Games, Yelp, Al-Nahal Al-Jawal

Exploratory analysis only: load each raw source, adapt it to a shared schema where that makes sense, check referential integrity, and report scale / density / text quality. No modeling or baselines here — see `model_comparison.ipynb` for the Item-CF vs LLM recommender comparison that follows from this EDA.

## 0. Setup — Kaggle input path
#### Three datasets were chosen to span different sparsity levels: Amazon Beauty (extremely sparse, ~1.1 interactions/user), Amazon Video Games (moderate density), and Yelp (different domain entirely validates the pipeline generalizes beyond one provider's data format). This diversity is intentional: it lets us test whether any result is dataset-specific or holds across data characteristics.

In [1]:
import pandas as pd 
import kagglehub
import numpy as np

base_path = "/kaggle/input/datasets/taherabed23/dataset/data/raw"

# --- Amazon Beauty ---
# Note: Kaggle auto-decompressed the .gz files, so no compression= argument needed here
df_reviews_beauty = pd.read_json(f"{base_path}/amazon_beauty/reviews.jsonl/All_Beauty.jsonl", lines=True)
df_meta_beauty = pd.read_json(f"{base_path}/amazon_beauty/meta.jsonl/meta_All_Beauty.jsonl", lines=True)

# --- Amazon Video Games ---
chunks_vg = pd.read_json(f"{base_path}/amazon_video_games/reviews.jsonl/Video_Games.jsonl", lines=True, chunksize=300_000)
df_reviews_vg = next(chunks_vg)
df_meta_vg = pd.read_json(f"{base_path}/amazon_video_games/meta.jsonl/meta_Video_Games.jsonl", lines=True)

# --- Yelp business (uploaded successfully) ---
df_business_yelp = pd.read_json(f"{base_path}/yelp/business.json", lines=True)

# --- Yelp reviews (missing from the upload) — download fresh directly on Kaggle ---
yelp_path = kagglehub.dataset_download("yelp-dataset/yelp-dataset")
chunks_yelp = pd.read_json(f"{yelp_path}/yelp_academic_dataset_review.json", lines=True, chunksize=300_000)
df_reviews_yelp = next(chunks_yelp)

print("Beauty:", df_reviews_beauty.shape, df_meta_beauty.shape)
print("Video Games:", df_reviews_vg.shape, df_meta_vg.shape)
print("Yelp:", df_reviews_yelp.shape, df_business_yelp.shape)

Beauty: (701528, 10) (112590, 14)
Video Games: (300000, 10) (137269, 16)
Yelp: (300000, 9) (150346, 14)


## 2. Canonical schema adapters
#### Each raw dataset uses different column names and structures (parent_asin vs business_id, millisecond timestamps vs date strings). Rather than duplicating downstream logic per dataset, each source gets a thin adapter that maps it to one shared schema (user_id, item_id, timestamp, item_text, rating). This means adding a 4th dataset in the future requires only a new adapter function zero changes to EDA, baselines, or the LLM pipeline. This mirrors the industry-standard "data contract" / Bronze-Silver-Gold pattern.

In [2]:
CANONICAL_COLUMNS = ["user_id", "item_id", "timestamp", "item_text", "rating"]


def adapt_amazon(reviews_df: pd.DataFrame, meta_df: pd.DataFrame) -> pd.DataFrame:
    """
    Adapt Amazon Reviews 2023 data to the canonical interaction schema.

    Mapping:
        user_id     -> user_id
        parent_asin -> item_id
        timestamp   -> timestamp (epoch milliseconds -> datetime)
        product title -> item_text
        rating      -> rating

    Product metadata is joined using parent_asin. The product title is
    retained as item_text; the review title is intentionally excluded.
    """
    merged = reviews_df.merge(
        meta_df[["parent_asin", "title"]],
        on="parent_asin",
        how="left",
        suffixes=("_review", "_product"),
    )
    canonical = pd.DataFrame({
        "user_id": merged["user_id"],
        "item_id": merged["parent_asin"],
        "timestamp": pd.to_datetime(merged["timestamp"], unit="ms", errors="coerce"),
        "item_text": merged["title_product"],
        "rating": merged["rating"],
    })
    canonical["source"] = "amazon"
    return canonical[CANONICAL_COLUMNS + ["source"]]


def adapt_yelp(reviews_df: pd.DataFrame, business_df: pd.DataFrame) -> pd.DataFrame:
    """
    Adapt Yelp review data to the canonical interaction schema.

    Mapping:
        user_id     -> user_id
        business_id -> item_id
        date        -> timestamp
        business name -> item_text
        stars       -> rating
    """
    merged = reviews_df.merge(
        business_df[["business_id", "name"]],
        on="business_id",
        how="left",
    )
    canonical = pd.DataFrame({
        "user_id": merged["user_id"],
        "item_id": merged["business_id"],
        "timestamp": pd.to_datetime(merged["date"], errors="coerce"),
        "item_text": merged["name"],
        "rating": merged["stars"],
    })
    canonical["source"] = "yelp"
    return canonical[CANONICAL_COLUMNS + ["source"]]


ADAPTERS = {"amazon": adapt_amazon, "yelp": adapt_yelp}

In [3]:
canonical_beauty = adapt_amazon(df_reviews_beauty, df_meta_beauty)
canonical_vg = adapt_amazon(df_reviews_vg, df_meta_vg)
canonical_yelp = adapt_yelp(df_reviews_yelp, df_business_yelp)

print(canonical_beauty.shape, canonical_vg.shape, canonical_yelp.shape)
canonical_beauty.head()

(701528, 6) (300000, 6) (300000, 6)


,user_id,item_id,timestamp,item_text,rating,source
0,AGKHLEW2SOWHNMFQIJGBECAF7INQ,B00YQ6X8EO,2020-05-05 14:08:48.923,Herbivore - Natural Sea Mist Texturizing Salt ...,5,amazon
1,AGKHLEW2SOWHNMFQIJGBECAF7INQ,B081TJ8YS3,2020-05-04 18:10:55.070,All Natural Vegan Dry Shampoo Powder - Eco Fri...,4,amazon
2,AE74DYR3QUGVPZJ3P7RFWBGIX7XQ,B097R46CSY,2020-05-16 21:41:06.052,New Road Beauty - Creamsicle - Variety 3 Pack ...,5,amazon
3,AFQLNQNQYFWQZPJQZS6V3NZU4QBQ,B09JS339BZ,2022-01-28 18:13:50.220,muaowig Ombre Body Wave Bundles 1B Grey Human ...,1,amazon
4,AFQLNQNQYFWQZPJQZS6V3NZU4QBQ,B08BZ63GMJ,2020-12-30 10:02:43.534,Yinhua Electric Nail Drill Kit Portable Profes...,5,amazon



### 3. Referential Integrity Check

Before building any downstream logic, we verify that every interaction references a valid item.

**Result:** All 3 datasets showed **100% referential integrity**.

This result was independently verified using an adversarial test: a fake item ID was injected, and the check correctly flagged it as unmatched. This provided evidence that the perfect match was genuine rather than the result of a faulty check.

In [4]:
def check_id_match(interactions_df, reference_df, id_col_interactions, id_col_reference):
    """
    Checks referential integrity: does every interaction point to an item
    that actually exists in the reference table? Works on any dataset since
    column names are passed as parameters, not hardcoded.
    """
    reference_ids = reference_df[id_col_reference].unique()
    matched_mask = interactions_df[id_col_interactions].isin(reference_ids)

    n_total = len(matched_mask)
    n_matched = int(matched_mask.sum())  # True=1, False=0
    n_orphan = n_total - n_matched

    return {
        "total": n_total,
        "matched": n_matched,
        "matched_pct": round(n_matched / n_total * 100, 2) if n_total else 0,
        "orphan": n_orphan,
        "orphan_pct": round(n_orphan / n_total * 100, 2) if n_total else 0,
    }


# Same function, different id column names per dataset — result: 100%
# referential integrity across all 3 (verified with a fake-id test first)
print("Beauty:", check_id_match(df_reviews_beauty, df_meta_beauty, "parent_asin", "parent_asin"))
print("Video Games:", check_id_match(df_reviews_vg, df_meta_vg, "parent_asin", "parent_asin"))
print("Yelp:", check_id_match(df_reviews_yelp, df_business_yelp, "business_id", "business_id"))

Beauty: {'total': 701528, 'matched': 701528, 'matched_pct': 100.0, 'orphan': 0, 'orphan_pct': 0.0}
Video Games: {'total': 300000, 'matched': 300000, 'matched_pct': 100.0, 'orphan': 0, 'orphan_pct': 0.0}
Yelp: {'total': 300000, 'matched': 300000, 'matched_pct': 100.0, 'orphan': 0, 'orphan_pct': 0.0}


> ### EDA / Suitability Report
>
> The EDA report evaluates both **dataset quality** and **task suitability** before downstream modeling.
>
> It measures:
> - **Dataset scale:** number of interactions, users, and items.
> - **User interaction density:** average and median interactions per user, plus the percentage of users meeting the minimum interaction threshold.
> - **Text suitability:** item-text coverage, average text length, and the percentage of texts below the minimum usable length.
> - **Rating validity:** minimum and maximum observed ratings.
> - **Interaction structure:** duplicate user-item pairs.
> - **Temporal coverage:** the span of valid timestamps in days.
>
> The thresholds `min_interactions=5` and `min_text_len=5` are configurable so that suitability criteria can be adjusted without changing the reporting logic.

In [5]:
def eda_report(canonical_df, min_interactions=5, min_text_len=5):
    report = {}

    # Dataset scale
    report["n_rows"] = len(canonical_df)
    report["n_users"] = canonical_df["user_id"].nunique()
    report["n_items"] = canonical_df["item_id"].nunique()

    # User interaction density
    per_user = canonical_df.groupby("user_id").size()
    report["avg_interactions_per_user"] = round(per_user.mean(), 3)
    report["median_interactions_per_user"] = per_user.median()
    report["users_meeting_min_interactions"] = int(
        (per_user >= min_interactions).sum()
    )
    report["pct_users_meeting_min_interactions"] = round(
        (per_user >= min_interactions).mean() * 100, 2
    )

    # Item-text suitability
    non_null_text = canonical_df["item_text"].notna()
    report["item_text_coverage_pct"] = round(non_null_text.mean() * 100, 2)

    text_lengths = canonical_df.loc[non_null_text, "item_text"].str.len()
    report["avg_item_text_length"] = (
        round(text_lengths.mean(), 1) if len(text_lengths) else 0
    )
    report["pct_item_text_too_short"] = (
        round((text_lengths < min_text_len).mean() * 100, 2)
        if len(text_lengths)
        else 100.0
    )

    # Rating and interaction integrity
    report["rating_min"] = canonical_df["rating"].min()
    report["rating_max"] = canonical_df["rating"].max()
    report["duplicate_user_item_pairs"] = int(
        canonical_df.duplicated(
            subset=["user_id", "item_id"]
        ).sum()
    )

    # Temporal coverage
    valid_ts = canonical_df["timestamp"].dropna()
    if len(valid_ts):
        report["timestamp_span_days"] = (
            valid_ts.max() - valid_ts.min()
        ).days
    else:
        report["timestamp_span_days"] = 0

    return report


def print_eda_report(name, report):
    print(f"\n===== EDA / Suitability Report: {name} =====")
    for k, v in report.items():
        print(f"{k:35s}: {v}")


eda_results = {}

for name, df in [
    ("Beauty", canonical_beauty),
    ("Video Games", canonical_vg),
    ("Yelp", canonical_yelp),
]:
    eda_results[name] = eda_report(df)
    print_eda_report(name, eda_results[name])


===== EDA / Suitability Report: Beauty =====
n_rows                             : 701528
n_users                            : 631986
n_items                            : 112565
avg_interactions_per_user          : 1.11
median_interactions_per_user       : 1.0
users_meeting_min_interactions     : 1620
pct_users_meeting_min_interactions : 0.26
item_text_coverage_pct             : 100.0
avg_item_text_length               : 115.2
pct_item_text_too_short            : 0.03
rating_min                         : 1
rating_max                         : 5
duplicate_user_item_pairs          : 7599
timestamp_span_days                : 8346

===== EDA / Suitability Report: Video Games =====
n_rows                             : 300000
n_users                            : 127339
n_items                            : 48109
avg_interactions_per_user          : 2.356
median_interactions_per_user       : 1.0
users_meeting_min_interactions     : 12555
pct_users_meeting_min_interactions : 9.86
item_text_cove

## Al-Nahal Al-Jawal — a 4th dataset, item catalog not interactions

Unlike Amazon and Yelp above, this dataset has no `user_id` and no `timestamp` — it is a **product catalog** scraped directly from one Salla store (see `scrape_store.py`), not a set of user interactions. The canonical interaction schema (`user_id, item_id, timestamp, item_text, rating`) used above does not apply here, so this section analyzes it on its own terms: field coverage in the raw JSON-LD, and text/rating/price/category quality in the processed table (`build_items.py` turns the raw file into this table).

In [ ]:
import json
import pandas as pd

RAW_PATH = "../data/raw/alnahal/products.jsonl"
PROCESSED_PATH = "../data/processed/items_alnahal.csv"

alnahal_raw = [json.loads(line) for line in open(RAW_PATH, encoding="utf-8") if line.strip()]
alnahal_products = [r["product"] for r in alnahal_raw]

print("rows:", len(alnahal_raw))
print("top-level keys:", sorted(alnahal_raw[0].keys()))

product_key_counts = pd.Series(
    [k for p in alnahal_products for k in p.keys()]
).value_counts()
print("\nproduct field coverage (out of", len(alnahal_products), "products):")
print(product_key_counts)


In [ ]:
alnahal_items = pd.read_csv(PROCESSED_PATH)

print("shape:", alnahal_items.shape)
print("\ndtypes:")
print(alnahal_items.dtypes)
print("\nnull counts:")
print(alnahal_items.isna().sum())
print("\nduplicate item_ids:", alnahal_items["item_id"].duplicated().sum())
print("\nrating range:", alnahal_items["rating"].min(), "-", alnahal_items["rating"].max())
print("price range:", alnahal_items["price"].min(), "-", alnahal_items["price"].max())


In [ ]:
desc_lengths = alnahal_items["description"].str.len()
print("description length - min/avg/max:", desc_lengths.min(), round(desc_lengths.mean(), 1), desc_lengths.max())

same_as_title = alnahal_items["description"] == alnahal_items["title"]

# Align by item_id, not row position: build_items.py shuffles rows
# (save() samples with random_state=42), so the processed CSV is not
# in the same order as the raw JSONL file.
raw_no_description = {
    int(r["item_id"]): not r["product"].get("description") for r in alnahal_raw
}
no_raw_description = alnahal_items["item_id"].map(raw_no_description)

print("items with description == title:", same_as_title.sum())
print("  of which, no description at all in the source (our fallback kicked in):", int((same_as_title & no_raw_description).sum()))
print("  of which, source already set description = name (merchant's own data):", int((same_as_title & ~no_raw_description).sum()))

print("\ncategory coverage:", alnahal_items["category"].notna().sum(), "/", len(alnahal_items))
print(alnahal_items["category"].value_counts())


**Takeaways:**
- `category` is present in the source for only 5 of 286 items — too sparse to rely on for filtering without a fallback.
- 32 items end up with `description == title`, but for two different reasons: 4 have no description at all in the source and fall back to the title (see `alnahal_items` in `build_items.py`), while the other 28 already have a `description` field in the store's own JSON-LD that the merchant simply set equal to the product name — a content-quality issue in the source itself, not something our pipeline introduced.
- `item_id` round-trips through the CSV as `int64` even though the source values are strings — safe today only because every id is purely numeric with no leading zero. Worth forcing `dtype={'item_id': str}` on read if a future dataset's ids are less uniform.
- No duplicate item_ids, and rating/price ranges are within expected bounds (1-5 stars, a few SAR up to a bit over 1000).